<center>
<img src="https://supportvectors.ai/logo-poster-transparent.png" width=400px style="opacity:0.8">
</center>

# Three-Layer Global Memory

Global memory co-evolves with the indexing graph:

- frequent schemas become **stable** and filter noise
- conflicting facts are detected and resolved using passage evidence

In [1]:
%run supportvectors-common.ipynb



<div style="color:#aaa;font-size:8pt">
<hr/>
&copy; SupportVectors. All rights reserved. <blockquote>This notebook is the intellectual property of SupportVectors, and part of its training material. 
Only the participants in SupportVectors workshops are allowed to study the notebooks for educational purposes currently, but is prohibited from copying or using it for any other purposes without written permission.

<b> These notebooks are chapters and sections from Asif Qamar's textbook that he is writing on Data Science. So we request you to not circulate the material to others.</b>
 </blockquote>
 <hr/>
</div>



In [2]:

from memg_concepts.config import DEFAULT_PDF
from memg_concepts.document import build_passages
from memg_concepts.extraction import extract_from_passage, to_fact_triples
from memg_concepts.memory import GlobalMemory

memory = GlobalMemory()
for passage in build_passages(DEFAULT_PDF, max_chunks=10):
    memory.add_passage(passage)
    extraction = extract_from_passage(passage)
    for fact in to_fact_triples(extraction.facts, passage.passage_id):
        memory.add_fact(fact)

conflicts = memory.detect_conflicts()
removed = memory.resolve_conflicts(conflicts)
# Build the schema layer programmatically from the surviving facts.
memory.derive_schemas()
print('Summary:', memory.summary())
print('Conflicts resolved:', len(removed))

Summary: {'passages': 10, 'schemas': 82, 'stable_schemas': 5, 'facts': 115, 'active_facts': 15}
Conflicts resolved: 17


In [3]:
stable = memory.stable_schemas(min_freq=2)
active = memory.active_facts(min_schema_freq=2)
print('Stable schemas:', list(stable.keys())[:8])
print('Active facts:', list(active.keys())[:8])

Stable schemas: ['(method, is, concept)', '(model, based_on, concept)', '(person, designed, concept)', '(person, implemented, concept)', '(model, trained_on, tool)']
Active facts: ['(Attention Is All You Need, is, research paper)', '(Transformer, based_on, attention mechanisms)', '(Ashish, designed, first Transformer models)', '(Ashish, implemented, first Transformer models)', '(Illia, designed, first Transformer models)', '(Illia, implemented, first Transformer models)', '(Niki, designed, model variants)', '(Niki, implemented, model variants)']


In [4]:
stable = memory.stable_schemas(min_freq=1)
active = memory.active_facts(min_schema_freq=1)
print('Stable schemas:', list(stable.keys())[:8])
print('Active facts:', list(active.keys())[:8])

Stable schemas: ['(organization, grants_permission_to_reproduce, concept)', '(method, is, concept)', '(person, is_author_of, method)', '(person, affiliated_with, organization)', '(model, is_a, concept)', '(model, based_on, concept)', '(model, dispenses_with, concept)', '(metric, on_task, task)']
Active facts: ['(Google, grants_permission_to_reproduce, tables and figures in this paper)', '(Attention Is All You Need, is, research paper)', '(Ashish Vaswani, is_author_of, Attention Is All You Need)', '(Ashish Vaswani, affiliated_with, Google Brain)', '(Noam Shazeer, is_author_of, Attention Is All You Need)', '(Noam Shazeer, affiliated_with, Google Brain)', '(Niki Parmar, is_author_of, Attention Is All You Need)', '(Niki Parmar, affiliated_with, Google Research)']
